# OMNet-V3: Magnification-Aware Multi-Task Fusion Framework for Breast Cancer Histopathology

**OMNet-V3** is a dual-branch deep learning architecture combining **EfficientNet-B0** (local morphological features) and **ViT-Tiny/16** (global context representation) with a **Magnification-Aware Adaptive Fusion (MAF)** gating mechanism and a **Hierarchical Multi-Task Loss** on the **BreakHis** dataset (all magnifications: 40X, 100X, 200X, 400X).

---

## Key Innovations

1. **Dual-Branch Pretrained Backbones:**
   - EfficientNet-B0 (1280-d local features) + ViT-Tiny/16 (192-d self-attention global context).
2. **Magnification-Aware Adaptive Fusion (MAF):**
   - Learns a scale-conditioned gating parameter $\alpha \in (0, 1)$ conditioned on the magnification embedding ($e_m \in \mathbb{R}^{64}$), dynamically balancing local CNN details and global ViT structure across 40X to 400X magnifications.
3. **Hierarchical Multi-Task Loss:**
   - Combines primary binary classification (benign vs. malignant, $L_{\text{binary}}$), 8-class histological subtype classification ($L_{\text{subtype}}$), and hierarchical probability consistency ($L_{\text{consistency}}$):
   $$L_{\text{total}} = 0.3 L_{\text{binary}} + 0.6 L_{\text{subtype}} + 0.1 L_{\text{consistency}}$$
4. **Patient-Disjoint 5-Fold Stratified Group Cross-Validation:**
   - Strict `StratifiedGroupKFold` on `patient_id` guaranteeing zero patient identity leakage across folds across all magnifications.
5. **Direct Fast SSD Cache Fetching & Drive Checkpointing:**
   - Direct download via `kagglehub` on local high-speed SSD cache and persistence of all checkpoints, metrics, and plots to `/content/drive/MyDrive/output_v3/`.

# Section 1: Environment Setup

In [1]:
# ============================================================
# Section 1: Environment Setup
# ============================================================
!pip install -q kagglehub timm>=0.9.0 scikit-learn seaborn tqdm grad-cam

import os
import sys
import json
import random
import math
import warnings
import gc
import zipfile
import shutil
import glob
from pathlib import Path
from datetime import datetime, timedelta
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, Subset
from torchvision import transforms
from torchvision.transforms import functional as TF
import torchvision.models as tvm
import timm

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    matthews_corrcoef,
    precision_recall_fscore_support,
    roc_curve,
    precision_recall_curve
)
from sklearn.manifold import TSNE

warnings.filterwarnings("ignore")

def set_seed(seed=42):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(42)

print("=" * 60)
print("  OMNet-V3 Environment Setup Complete")
print("=" * 60)
print(f"  PyTorch     : {torch.__version__}")
print(f"  timm        : {timm.__version__}")
print(f"  CUDA Avail  : {torch.cuda.is_available()}")
print("=" * 60)

  OMNet-V3 Environment Setup Complete
  PyTorch     : 2.11.0+cu128
  timm        : 1.0.28
  CUDA Avail  : True


# Section 2: GPU Detection

In [2]:
# ============================================================
# Section 2: GPU Detection
# ============================================================
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("=" * 60)
print("  Hardware Configuration")
print("=" * 60)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"  Device         : {torch.cuda.get_device_name(0)}")
    print(f"  Compute Cap.   : {props.major}.{props.minor}")
    print(f"  Total Memory   : {props.total_memory / 1e9:.2f} GB")
    print(f"  Multiprocessors: {props.multi_processor_count}")
else:
    print("  [WARNING] No GPU detected. Execution will proceed on CPU.")
print(f"  Active Device  : {DEVICE}")
print("=" * 60)

  Hardware Configuration
  Device         : Tesla T4
  Compute Cap.   : 7.5
  Total Memory   : 15.64 GB
  Multiprocessors: 40
  Active Device  : cuda


# Section 3: Kaggle Authentication & Google Drive Mount

In [3]:
# ============================================================
# Section 3: Kaggle Authentication & Google Drive Mount
# ============================================================
# Kaggle API Authentication Token
os.environ["KAGGLE_API_TOKEN"] = "KGAT_18c71d28310cd8321b7765aaceb54d3e"
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = Path('/content/drive/MyDrive/output_v3')
else:
    OUTPUT_DIR = Path('./output_v3')

PROJECT_ROOT = OUTPUT_DIR

# --- Output subdirectories ---
GRADCAM_DIR = OUTPUT_DIR / 'gradcam'
SCORECAM_DIR = OUTPUT_DIR / 'scorecam'
MISCLASSIFIED_DIR = OUTPUT_DIR / 'misclassified'
CORRECT_PRED_DIR = OUTPUT_DIR / 'correct_predictions'

for d in [OUTPUT_DIR, GRADCAM_DIR, SCORECAM_DIR, MISCLASSIFIED_DIR, CORRECT_PRED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

for fold_idx in range(5):
    (OUTPUT_DIR / f'fold_{fold_idx}').mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("  Storage & Authentication Configuration")
print("=" * 60)
print(f"  Environment  : {'Google Colab' if IN_COLAB else 'Local Machine'}")
print(f"  Output Dir   : {OUTPUT_DIR}")
print(f"  Kaggle Auth  : KAGGLE_API_TOKEN active")
print("=" * 60)

def save_figure(fig, filename, dpi=150):
    path = OUTPUT_DIR / filename
    fig.savefig(path, dpi=dpi, bbox_inches='tight', facecolor='white')
    print(f"  [SAVE] Saved: {path}")

Mounted at /content/drive
  Storage & Authentication Configuration
  Environment  : Google Colab
  Output Dir   : /content/drive/MyDrive/output_v3
  Kaggle Auth  : KAGGLE_API_TOKEN active


# Centralized Configuration

In [4]:
# ============================================================
# Centralized Configuration
# ============================================================

CONFIG = {
    # --- Data ---
    "dataset_id": "trexbytes/breakhislink",
    "output_dir": str(OUTPUT_DIR),
    "image_size": 224,
    "batch_size": 16,
    "num_workers": 0,
    "n_splits": 5,
    "magnification_levels": [40, 100, 200, 400],
    "binary_classes": ["benign", "malignant"],
    "subtype_order": ["A", "F", "PT", "TA", "DC", "LC", "MC", "PC"],
    "subtype_to_index": {"A": 0, "F": 1, "PT": 2, "TA": 3, "DC": 4, "LC": 5, "MC": 6, "PC": 7},
    "binary_mapping": {"A": 0, "F": 0, "PT": 0, "TA": 0, "DC": 1, "LC": 1, "MC": 1, "PC": 1},
    "train_resize": 256,
    "train_crop": 224,
    "val_resize": 256,
    "val_crop": 224,
    "stain_aug_prob": 0.5,

    # --- Model ---
    "fusion_dim": 256,
    "magnification_embedding_dim": 64,
    "dropout": 0.3,
    "cnn_backbone": "efficientnet_b0",
    "vit_backbone": "vit_tiny_patch16_224",

    # --- Training ---
    "max_epochs": 30,
    "warmup_epochs": 5,
    "early_stopping_patience": 8,
    "base_lr": 1e-4,
    "head_lr": 5e-4,
    "weight_decay": 1e-4,
    "gradient_clip_norm": 1.0,
    "amp_enabled": True,
    "loss_weights": {"binary": 0.3, "subtype": 0.6, "consistency": 0.1},

    # --- Reproducibility ---
    "seed": 42,
    "smoke_test": False,
}

if CONFIG['smoke_test']:
    CONFIG['n_splits'] = 2
    CONFIG['max_epochs'] = 2
    CONFIG['warmup_epochs'] = 1

set_seed(CONFIG['seed'])

print("=" * 60)
print("  Experiment Configuration (OMNet-V3)")
print("=" * 60)
for k, v in CONFIG.items():
    print(f"  {k:30s}: {v}")
print("=" * 60)

  Experiment Configuration (OMNet-V3)
  dataset_id                    : trexbytes/breakhislink
  output_dir                    : /content/drive/MyDrive/output_v3
  image_size                    : 224
  batch_size                    : 16
  num_workers                   : 0
  n_splits                      : 5
  magnification_levels          : [40, 100, 200, 400]
  binary_classes                : ['benign', 'malignant']
  subtype_order                 : ['A', 'F', 'PT', 'TA', 'DC', 'LC', 'MC', 'PC']
  subtype_to_index              : {'A': 0, 'F': 1, 'PT': 2, 'TA': 3, 'DC': 4, 'LC': 5, 'MC': 6, 'PC': 7}
  binary_mapping                : {'A': 0, 'F': 0, 'PT': 0, 'TA': 0, 'DC': 1, 'LC': 1, 'MC': 1, 'PC': 1}
  train_resize                  : 256
  train_crop                    : 224
  val_resize                    : 256
  val_crop                      : 224
  stain_aug_prob                : 0.5
  fusion_dim                    : 256
  magnification_embedding_dim   : 64
  dropout              

# Section 4: Dataset Download & Discovery via kagglehub

In [5]:
# ============================================================
# Section 4: Dataset Download & Discovery via kagglehub
# ============================================================
import kagglehub

dataset_id = CONFIG['dataset_id']
print(f"[INFO] Downloading dataset '{dataset_id}' directly via kagglehub...")
download_path = kagglehub.dataset_download(dataset_id)
print(f"[INFO] Download path: {download_path}")

local_extract_dir = Path('./data_breakhis')
zip_files = glob.glob(os.path.join(download_path, '**', '*.zip'), recursive=True)

if zip_files:
    os.makedirs(local_extract_dir, exist_ok=True)
    for zf in zip_files:
        print(f"[INFO] Extracting archive {zf} to {local_extract_dir}...")
        with zipfile.ZipFile(zf, 'r') as z:
            z.extractall(local_extract_dir)
    search_root = local_extract_dir
else:
    search_root = Path(download_path)

print(f"[INFO] Scanning {search_root} for PNG files...")

def parse_breakhis_filename(path: str) -> Optional[dict]:
    filename = os.path.basename(path)
    if not filename.lower().endswith('.png'):
        return None

    fname_no_ext = filename.replace('.png', '')
    parts = fname_no_ext.split('-')

    if len(parts) < 4:
        return None

    try:
        prefix_parts = parts[0].split('_')
        if len(prefix_parts) < 3:
            return None

        raw_class = prefix_parts[1]
        raw_subtype = prefix_parts[2]

        mag_token = parts[-2]
        magnification = int(mag_token)

        if magnification not in CONFIG['magnification_levels']:
            return None

        class_name = 'benign' if raw_class == 'B' else 'malignant'
        subtype = raw_subtype

        if subtype not in CONFIG['subtype_to_index']:
            return None

        case_id = '-'.join(parts[1:-2])
        patient_id = f"{raw_subtype}_{case_id}"
        seq = int(parts[-1])

        return {
            'file_path': path,
            'filename': filename,
            'class_name': class_name,
            'subtype': subtype,
            'subtype_index': CONFIG['subtype_to_index'][subtype],
            'binary_label': CONFIG['binary_mapping'][subtype],
            'magnification': magnification,
            'magnification_index': CONFIG['magnification_levels'].index(magnification),
            'patient_id': patient_id,
            'sequence': seq,
        }
    except Exception:
        return None

records = []
for full_path in Path(search_root).rglob('*.png'):
    parsed = parse_breakhis_filename(str(full_path))
    if parsed is not None:
        records.append(parsed)

assert len(records) > 0, f"[ERROR] No valid BreaKHis images found in {search_root}!"

metadata = pd.DataFrame(records)
print(f"[OK] Discovered {len(metadata)} total images across {metadata['patient_id'].nunique()} patients.")

print("\n--- Distribution by Class ---")
print(metadata['class_name'].value_counts())
print("\n--- Distribution by Subtype ---")
print(metadata['subtype'].value_counts())
print("\n--- Distribution by Magnification ---")
print(metadata['magnification'].value_counts())
print("\n--- Patients per Subtype ---")
print(metadata.groupby('subtype')['patient_id'].nunique())

[INFO] Downloading dataset 'trexbytes/breakhislink' directly via kagglehub...


100%|██████████| 3.98G/3.98G [03:16<00:00, 21.8MB/s]

Extracting files...


[INFO] Download path: /root/.cache/kagglehub/datasets/trexbytes/breakhislink/versions/1
[INFO] Scanning /root/.cache/kagglehub/datasets/trexbytes/breakhislink/versions/1 for PNG files...
[OK] Discovered 7909 total images across 82 patients.

--- Distribution by Class ---
class_name
malignant    5429
benign       2480
Name: count, dtype: int64

--- Distribution by Subtype ---
subtype
DC    3451
F     1014
MC     792
LC     626
TA     569
PC     560
PT     453
A      444
Name: count, dtype: int64

--- Distribution by Magnification ---
magnification
100    2081
200    2013
40     1995
400    1820
Name: count, dtype: int64

--- Patients per Subtype ---
subtype
A      4
DC    38
F     10
LC     5
MC     9
PC     6
PT     3
TA     7
Name: patient_id, dtype: int64


# Section 5: Patient-Disjoint Splitting (StratifiedGroupKFold)

In [6]:
# ============================================================
# Section 5: Patient-Disjoint Splitting (StratifiedGroupKFold)
# ============================================================
X = metadata[['file_path', 'subtype', 'patient_id']].copy()
y = metadata['subtype_index'].values
groups = metadata['patient_id'].values

sgkf = StratifiedGroupKFold(n_splits=CONFIG['n_splits'], shuffle=True, random_state=CONFIG['seed'])
split_indices = list(sgkf.split(X, y, groups))

print(f"[OK] Created {len(split_indices)} patient-disjoint folds.")

for fold_idx, (train_idx, test_idx) in enumerate(split_indices):
    train_df = metadata.iloc[train_idx]
    test_df = metadata.iloc[test_idx]
    train_p = set(train_df['patient_id'])
    test_p = set(test_df['patient_id'])
    overlap = train_p & test_p
    assert len(overlap) == 0, f"[ERROR] Patient leakage in Fold {fold_idx}: {overlap}"
    print(f"  Fold {fold_idx}: train={len(train_df):4d} ({len(train_p):2d} patients) | test={len(test_df):4d} ({len(test_p):2d} patients)")

# Save split overview
metadata.to_csv(OUTPUT_DIR / 'dataset_metadata.csv', index=False)
print(f"[SAVE] Exported dataset_metadata.csv to {OUTPUT_DIR}")

[OK] Created 5 patient-disjoint folds.
  Fold 0: train=6345 (66 patients) | test=1564 (16 patients)
  Fold 1: train=6418 (66 patients) | test=1491 (16 patients)
  Fold 2: train=6257 (66 patients) | test=1652 (16 patients)
  Fold 3: train=6270 (64 patients) | test=1639 (18 patients)
  Fold 4: train=6346 (66 patients) | test=1563 (16 patients)
[SAVE] Exported dataset_metadata.csv to /content/drive/MyDrive/output_v3


# Section 6: Data Augmentations & Stain Perturbation

In [7]:
# ============================================================
# Section 6: Data Augmentations & Stain Perturbation
# ============================================================

def rgb_to_od(img):
    img = img.astype(np.float32) / 255.0
    eps = 1e-6
    img = np.clip(img, eps, 1.0)
    return -np.log(img)

def estimate_he_vectors(od):
    od = od.reshape(-1, 3)
    od = od[~np.any(np.isinf(od) | np.isnan(od), axis=1)]
    if od.shape[0] == 0:
        return np.eye(3)
    _, _, vh = np.linalg.svd(od, full_matrices=False)
    return vh[:2, :]

def macenko_perturb(img: np.ndarray, p: float = 0.5):
    if np.random.rand() > p:
        return img
    arr = np.asarray(img)
    if arr.ndim != 3 or arr.shape[-1] != 3:
        return img
    od = rgb_to_od(arr)
    he_basis = estimate_he_vectors(od)
    stain = np.dot(od.reshape(-1, 3), he_basis.T)
    stain = stain.reshape(arr.shape[0], arr.shape[1], 2)
    if stain.size == 0:
        return img
    rand_factors = np.random.uniform(0.85, 1.15, size=(2,))
    rand_bias = np.random.uniform(-0.05, 0.05, size=(2,))
    perturbed = stain * rand_factors + rand_bias
    recon = np.dot(perturbed.reshape(-1, 2), he_basis).reshape(arr.shape[0], arr.shape[1], 3)
    recon = np.clip(recon, 0.0, 1.0)
    return (recon * 255.0).astype(np.uint8)

print("[OK] Macenko stain perturbation module initialized.")

[OK] Macenko stain perturbation module initialized.


# Section 7: Dataset Class & DataLoaders

In [8]:
# ============================================================
# Section 7: Dataset Class & DataLoaders
# ============================================================

class BreakHisDataset(Dataset):
    def __init__(self, df: pd.DataFrame, split: str = 'train', transform=None):
        self.df = df.reset_index(drop=True)
        self.split = split
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['file_path']).convert('RGB')

        if self.transform is not None:
            image = self.transform(image)

        return {
            'image': image,
            'binary_label': int(row['binary_label']),
            'subtype_label': int(row['subtype_index']),
            'magnification_index': int(row['magnification_index']),
            'patient_id': row['patient_id'],
            'file_path': row['file_path'],
        }

def build_transforms(split: str):
    if split == 'train':
        return transforms.Compose([
            transforms.Resize((CONFIG['train_resize'], CONFIG['train_resize'])),
            transforms.RandomResizedCrop(CONFIG['train_crop'], scale=(0.8, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(20),
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ])
    return transforms.Compose([
        transforms.Resize((CONFIG['val_resize'], CONFIG['val_resize'])),
        transforms.CenterCrop(CONFIG['val_crop']),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ])

def custom_train_transform(image: Image.Image):
    image = transforms.Resize((CONFIG['train_resize'], CONFIG['train_resize']))(image)
    image = transforms.RandomResizedCrop(CONFIG['train_crop'], scale=(0.8, 1.0))(image)
    image = transforms.RandomHorizontalFlip(p=0.5)(image)
    image = transforms.RandomVerticalFlip(p=0.5)(image)
    image = transforms.RandomRotation(20)(image)

    if np.random.rand() < CONFIG['stain_aug_prob']:
        image_np = np.array(image)
        image_np = macenko_perturb(image_np, p=1.0)
        image = Image.fromarray(image_np.astype(np.uint8))

    image = transforms.ToTensor()(image)
    image = transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))(image)
    return image

def build_dataloaders(train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame):
    train_ds = BreakHisDataset(train_df, 'train', custom_train_transform)
    val_ds = BreakHisDataset(val_df, 'val', build_transforms('val'))
    test_ds = BreakHisDataset(test_df, 'test', build_transforms('val'))

    nw = CONFIG['num_workers']  # 0 per stability guidelines
    train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=nw, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=nw, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=nw, pin_memory=True)
    return train_loader, val_loader, test_loader

print("[OK] Dataset and DataLoaders configured (num_workers=0).")

[OK] Dataset and DataLoaders configured (num_workers=0).


# Section 8: OMNet-V3 Architecture (Magnification-Aware Adaptive Fusion)

In [9]:
# ============================================================
# Section 8: OMNet-V3 Architecture
# ============================================================

class MagnificationAwareFusion(nn.Module):
    """
    Magnification-Aware Adaptive Fusion (MAF) Module:
    Dynamically balances CNN local features and ViT global context
    conditioned on magnification scale embedding.
    """
    def __init__(self, feat_dim=256, mag_dim=64):
        super().__init__()
        self.magnification_embedding = nn.Embedding(4, mag_dim)
        self.gate = nn.Sequential(
            nn.Linear(feat_dim * 2 + mag_dim, 1),
            nn.Sigmoid(),
        )
        self.mlp = nn.Sequential(
            nn.Linear(feat_dim + mag_dim, feat_dim),
            nn.GELU(),
            nn.Dropout(CONFIG['dropout']),
            nn.Linear(feat_dim, feat_dim),
        )

    def forward(self, f_cnn, f_vit, magnification_index):
        e_m = self.magnification_embedding(magnification_index)
        concat = torch.cat([f_cnn, f_vit, e_m], dim=-1)
        alpha = self.gate(concat)
        fused = alpha * f_cnn + (1.0 - alpha) * f_vit
        residual = self.mlp(torch.cat([fused, e_m], dim=-1))
        out = fused + residual
        return out, alpha


class OMNetV3(nn.Module):
    """
    OMNet-V3 Dual-Branch Architecture:
      - Branch 1: EfficientNet-B0 (1280-d local morphology)
      - Branch 2: ViT-Tiny/16 (192-d global context)
      - Fusion: Magnification-Aware Adaptive Fusion (256-d)
      - Classification Heads: Binary (2-class) + Subtype (8-class)
    """
    def __init__(self):
        super().__init__()
        self.cnn_backbone = timm.create_model(CONFIG['cnn_backbone'], pretrained=True, num_classes=0)
        self.vit_backbone = timm.create_model(CONFIG['vit_backbone'], pretrained=True, num_classes=0)

        self.cnn_proj = nn.Sequential(
            nn.Linear(1280, CONFIG['fusion_dim']),
            nn.LayerNorm(CONFIG['fusion_dim']),
        )
        self.vit_proj = nn.Sequential(
            nn.Linear(192, CONFIG['fusion_dim']),
            nn.LayerNorm(CONFIG['fusion_dim']),
        )
        self.fusion = MagnificationAwareFusion(
            feat_dim=CONFIG['fusion_dim'],
            mag_dim=CONFIG['magnification_embedding_dim']
        )
        self.binary_head = nn.Linear(CONFIG['fusion_dim'], 2)
        self.subtype_head = nn.Linear(CONFIG['fusion_dim'], 8)

    def forward_cnn(self, x):
        x = self.cnn_backbone.forward_features(x)
        x = self.cnn_backbone.global_pool(x)
        if x.dim() == 4:
            x = x.flatten(1)
        return self.cnn_proj(x)

    def forward_vit(self, x):
        x = self.vit_backbone.forward_features(x)
        if isinstance(x, (tuple, list)):
            x = x[0]
        if x.dim() == 3:
            x = x[:, 0, :]
        return self.vit_proj(x)

    def forward(self, x, magnification_index):
        f_cnn = self.forward_cnn(x)
        f_vit = self.forward_vit(x)
        f_out, alpha = self.fusion(f_cnn, f_vit, magnification_index)
        binary_logits = self.binary_head(f_out)
        subtype_logits = self.subtype_head(f_out)
        return {
            'f_out': f_out,
            'alpha': alpha,
            'binary_logits': binary_logits,
            'subtype_logits': subtype_logits,
        }

# Model sanity check
model_check = OMNetV3().to(DEVICE)
dummy_img = torch.randn(2, 3, 224, 224).to(DEVICE)
dummy_mag = torch.tensor([0, 2]).to(DEVICE)
with torch.no_grad():
    out_check = model_check(dummy_img, dummy_mag)
print("[OK] OMNet-V3 initialized successfully.")
print(f"     Binary logits shape  : {out_check['binary_logits'].shape}")
print(f"     Subtype logits shape : {out_check['subtype_logits'].shape}")
print(f"     Fusion embedding dim : {out_check['f_out'].shape}")
del model_check, dummy_img, dummy_mag, out_check
torch.cuda.empty_cache() if DEVICE.type == 'cuda' else None

model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 22.9MB            

model.safetensors: downloading bytes:           |  0.00B            

[OK] OMNet-V3 initialized successfully.
     Binary logits shape  : torch.Size([2, 2])
     Subtype logits shape : torch.Size([2, 8])
     Fusion embedding dim : torch.Size([2, 256])


# Section 9: Hierarchical Multi-Task Loss

In [10]:
# ============================================================
# Section 9: Hierarchical Multi-Task Loss
# ============================================================

class HierarchicalLoss(nn.Module):
    def __init__(self, class_weights_binary: torch.Tensor, class_weights_subtype: torch.Tensor):
        super().__init__()
        self.loss_binary = nn.CrossEntropyLoss(weight=class_weights_binary)
        self.loss_subtype = nn.CrossEntropyLoss(weight=class_weights_subtype)

    def forward(self, binary_logits, subtype_logits, binary_targets, subtype_targets):
        L_binary = self.loss_binary(binary_logits, binary_targets)
        L_subtype = self.loss_subtype(subtype_logits, subtype_targets)

        subtype_probs = torch.softmax(subtype_logits, dim=1)
        grouped = torch.stack([
            subtype_probs[:, :4].sum(dim=1),
            subtype_probs[:, 4:8].sum(dim=1),
        ], dim=1)
        consistency_target = torch.softmax(binary_logits, dim=1)
        L_consistency = F.kl_div(
            torch.log_softmax(grouped, dim=1),
            consistency_target,
            reduction='batchmean',
            log_target=False,
        )

        w = CONFIG['loss_weights']
        total = w['binary'] * L_binary + w['subtype'] * L_subtype + w['consistency'] * L_consistency
        return total, {'binary': L_binary, 'subtype': L_subtype, 'consistency': L_consistency}

def compute_class_weights(train_df: pd.DataFrame):
    n_total = len(train_df)
    binary_counts = train_df['binary_label'].value_counts().sort_index()
    subtype_counts = train_df['subtype_index'].value_counts().sort_index()

    binary_weights = torch.tensor([
        n_total / (2 * max(binary_counts.get(i, 1), 1)) for i in range(2)
    ], dtype=torch.float32)
    subtype_weights = torch.tensor([
        n_total / (8 * max(subtype_counts.get(i, 1), 1)) for i in range(8)
    ], dtype=torch.float32)
    return binary_weights, subtype_weights

print("[OK] Hierarchical multi-task loss module configured.")

[OK] Hierarchical multi-task loss module configured.


# Section 10: Training Engine (Modern torch.amp)

In [11]:
# ============================================================
# Section 10: Training Engine (Modern torch.amp)
# ============================================================

def set_requires_grad(model, flag):
    for p in model.parameters():
        p.requires_grad = flag

def train_one_epoch(model, loader, optimizer, scaler, criterion, device):
    model.train()
    running_loss = 0.0
    preds_binary, preds_subtype = [], []
    labels_binary, labels_subtype = [], []
    use_amp = CONFIG['amp_enabled'] and device.type == 'cuda'

    for batch in tqdm(loader, leave=False):
        images = batch['image'].to(device)
        binary_targets = batch['binary_label'].to(device)
        subtype_targets = batch['subtype_label'].to(device)
        magnification_idx = batch['magnification_index'].to(device)

        optimizer.zero_grad(set_to_none=True)
        if use_amp:
            with torch.amp.autocast('cuda', enabled=use_amp):
                outputs = model(images, magnification_idx)
                loss, _ = criterion(
                    outputs['binary_logits'],
                    outputs['subtype_logits'],
                    binary_targets,
                    subtype_targets,
                )
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['gradient_clip_norm'])
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images, magnification_idx)
            loss, _ = criterion(
                outputs['binary_logits'],
                outputs['subtype_logits'],
                binary_targets,
                subtype_targets,
            )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['gradient_clip_norm'])
            optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds_binary.append(torch.argmax(outputs['binary_logits'], dim=1).cpu())
        preds_subtype.append(torch.argmax(outputs['subtype_logits'], dim=1).cpu())
        labels_binary.append(binary_targets.cpu())
        labels_subtype.append(subtype_targets.cpu())

    epoch_loss = running_loss / max(len(loader.dataset), 1)
    binary_pred = torch.cat(preds_binary).numpy()
    subtype_pred = torch.cat(preds_subtype).numpy()
    binary_true = torch.cat(labels_binary).numpy()
    subtype_true = torch.cat(labels_subtype).numpy()

    return {
        'loss': epoch_loss,
        'binary_accuracy': accuracy_score(binary_true, binary_pred),
        'subtype_accuracy': accuracy_score(subtype_true, subtype_pred),
        'subtype_macro_f1': f1_score(subtype_true, subtype_pred, average='macro', zero_division=0),
    }

@torch.no_grad()
def evaluate_model(model, loader, device):
    model.eval()
    logits_binary, logits_subtype = [], []
    labels_binary, labels_subtype = [], []
    patient_ids, files = [], []
    use_amp = CONFIG['amp_enabled'] and device.type == 'cuda'

    for batch in loader:
        images = batch['image'].to(device)
        magnification_idx = batch['magnification_index'].to(device)

        if use_amp:
            with torch.amp.autocast('cuda', enabled=use_amp):
                outputs = model(images, magnification_idx)
        else:
            outputs = model(images, magnification_idx)

        logits_binary.append(outputs['binary_logits'].cpu())
        logits_subtype.append(outputs['subtype_logits'].cpu())
        labels_binary.append(batch['binary_label'])
        labels_subtype.append(batch['subtype_label'])
        patient_ids.extend(batch['patient_id'])
        files.extend(batch['file_path'])

    binary_probs = torch.softmax(torch.cat(logits_binary), dim=1).numpy()
    subtype_probs = torch.softmax(torch.cat(logits_subtype), dim=1).numpy()
    binary_preds = np.argmax(binary_probs, axis=1)
    subtype_preds = np.argmax(subtype_probs, axis=1)
    binary_true = torch.cat(labels_binary).numpy()
    subtype_true = torch.cat(labels_subtype).numpy()

    return {
        'binary_accuracy': accuracy_score(binary_true, binary_preds),
        'binary_macro_f1': f1_score(binary_true, binary_preds, average='macro', zero_division=0),
        'binary_balanced_accuracy': balanced_accuracy_score(binary_true, binary_preds),
        'binary_mcc': matthews_corrcoef(binary_true, binary_preds),
        'subtype_accuracy': accuracy_score(subtype_true, subtype_preds),
        'subtype_macro_f1': f1_score(subtype_true, subtype_preds, average='macro', zero_division=0),
        'subtype_weighted_f1': f1_score(subtype_true, subtype_preds, average='weighted', zero_division=0),
        'subtype_balanced_accuracy': balanced_accuracy_score(subtype_true, subtype_preds),
        'subtype_mcc': matthews_corrcoef(subtype_true, subtype_preds),
        'binary_confusion_matrix': confusion_matrix(binary_true, binary_preds).tolist(),
        'subtype_confusion_matrix': confusion_matrix(subtype_true, subtype_preds).tolist(),
        'subtype_probs': subtype_probs.tolist(),
        'binary_probs': binary_probs.tolist(),
        'subtype_preds': subtype_preds.tolist(),
        'binary_preds': binary_preds.tolist(),
        'subtype_true': subtype_true.tolist(),
        'binary_true': binary_true.tolist(),
        'patient_ids': patient_ids,
        'files': files,
    }

print("[OK] Training & evaluation functions ready.")

[OK] Training & evaluation functions ready.


# Section 11: 5-Fold Cross-Validation Execution

In [ ]:
# ============================================================
# Section 11: 5-Fold Cross-Validation Execution
# ============================================================

import time

def run_fold(fold_idx: int, full_df: pd.DataFrame):
    fold_output_dir = OUTPUT_DIR / f'fold_{fold_idx}'
    fold_output_dir.mkdir(parents=True, exist_ok=True)

    train_idx, test_idx = split_indices[fold_idx]
    train_df_full = full_df.iloc[train_idx].copy().reset_index(drop=True)
    test_df = full_df.iloc[test_idx].copy().reset_index(drop=True)

    train_patients = set(train_df_full['patient_id'])
    val_sample_count = max(1, int(0.10 * len(train_patients)))
    val_patients = set(pd.Series(list(train_patients)).sample(val_sample_count, random_state=CONFIG['seed'] + fold_idx))

    val_df = train_df_full[train_df_full['patient_id'].isin(val_patients)].copy().reset_index(drop=True)
    train_df = train_df_full[~train_df_full['patient_id'].isin(val_patients)].copy().reset_index(drop=True)

    binary_weights, subtype_weights = compute_class_weights(train_df)
    criterion = HierarchicalLoss(binary_weights.to(DEVICE), subtype_weights.to(DEVICE)).to(DEVICE)

    train_loader, val_loader, test_loader = build_dataloaders(train_df, val_df, test_df)

    model = OMNetV3().to(DEVICE)

    base_params = list(model.cnn_backbone.parameters()) + list(model.vit_backbone.parameters())
    head_params = (list(model.cnn_proj.parameters()) + list(model.vit_proj.parameters()) +
                   list(model.fusion.parameters()) + list(model.binary_head.parameters()) +
                   list(model.subtype_head.parameters()))

    optimizer = torch.optim.AdamW([
        {'params': base_params, 'lr': CONFIG['base_lr']},
        {'params': head_params, 'lr': CONFIG['head_lr']},
    ], weight_decay=CONFIG['weight_decay'])

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['max_epochs'])
    scaler = torch.amp.GradScaler('cuda', enabled=(CONFIG['amp_enabled'] and DEVICE.type == 'cuda')) if DEVICE.type == 'cuda' else None

    best_val_f1 = -1.0
    best_state = None
    history = []
    patience_counter = 0

    print(f"\n--- Fold {fold_idx}: Train={len(train_df)} imgs ({train_df['patient_id'].nunique()} pats) | Val={len(val_df)} imgs ({val_df['patient_id'].nunique()} pats) | Test={len(test_df)} imgs ({test_df['patient_id'].nunique()} pats) ---")

    for epoch in range(1, CONFIG['max_epochs'] + 1):
        epoch_start = time.time()
        # Progressive unfreezing: warmup head first, unfreeze backbone after warmup
        if epoch <= CONFIG['warmup_epochs']:
            set_requires_grad(model.cnn_backbone, False)
            set_requires_grad(model.vit_backbone, False)
        else:
            set_requires_grad(model.cnn_backbone, True)
            set_requires_grad(model.vit_backbone, True)

        train_metrics = train_one_epoch(model, train_loader, optimizer, scaler, criterion, DEVICE)
        val_metrics = evaluate_model(model, val_loader, DEVICE)
        scheduler.step()
        epoch_time = time.time() - epoch_start

        record = {
            'epoch': epoch,
            'train_loss': round(train_metrics['loss'], 4),
            'train_subtype_f1': round(train_metrics['subtype_macro_f1'], 4),
            'val_binary_f1': round(val_metrics['binary_macro_f1'], 4),
            'val_subtype_macro_f1': round(val_metrics['subtype_macro_f1'], 4),
            'val_subtype_acc': round(val_metrics['subtype_accuracy'], 4),
        }
        history.append(record)

        print(f"  Epoch {epoch:2d} | {epoch_time:4.1f}s | Train Loss: {record['train_loss']:.4f} | Val Subtype F1: {record['val_subtype_macro_f1']:.4f} | Val Bin F1: {record['val_binary_f1']:.4f}")

        if val_metrics['subtype_macro_f1'] > best_val_f1:
            best_val_f1 = val_metrics['subtype_macro_f1']
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            torch.save(best_state, fold_output_dir / 'best_model.pth')
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= CONFIG['early_stopping_patience'] and not CONFIG['smoke_test']:
                print(f"  [INFO] Early stopping triggered at epoch {epoch}.")
                break

    # Save training history
    with open(fold_output_dir / 'training_history.json', 'w') as f:
        json.dump(history, f, indent=2)

    # Evaluate best model state on held-out test set
    if best_state is not None:
        model.load_state_dict(best_state)
    model.to(DEVICE)
    test_metrics = evaluate_model(model, test_loader, DEVICE)

    with open(fold_output_dir / 'test_metrics.json', 'w') as f:
        json.dump(test_metrics, f, indent=2)

    del model, optimizer, train_loader, val_loader, test_loader
    gc.collect()
    torch.cuda.empty_cache() if DEVICE.type == 'cuda' else None

    return test_metrics, history

all_fold_results = []
for fold_idx in range(CONFIG['n_splits']):
    print(f"\n============================================================")
    print(f"  Running Patient-Disjoint Fold {fold_idx + 1}/{CONFIG['n_splits']}")
    print(f"============================================================")
    test_metrics, history = run_fold(fold_idx, metadata)
    all_fold_results.append({
        'fold_idx': fold_idx,
        'test_metrics': test_metrics,
        'history': history,
    })
    print(f"[OK] Fold {fold_idx} finished. Test Subtype Macro-F1: {test_metrics['subtype_macro_f1']:.4f} | Binary Acc: {test_metrics['binary_accuracy']:.4f}")


  Running Patient-Disjoint Fold 1/5

--- Fold 0: Train=5622 imgs (60 pats) | Val=723 imgs (6 pats) | Test=1564 imgs (16 pats) ---


  0%|          | 0/352 [00:00<?, ?it/s]

  Epoch  1 | 195.1s | Train Loss: 1.0675 | Val Subtype F1: 0.1685 | Val Bin F1: 0.8210


  0%|          | 0/352 [00:00<?, ?it/s]

  Epoch  2 | 158.0s | Train Loss: 0.8016 | Val Subtype F1: 0.1895 | Val Bin F1: 0.8110


...

  0%|          | 0/352 [00:00<?, ?it/s]

  Epoch  9 | 185.8s | Train Loss: 0.3061 | Val Subtype F1: 0.1498 | Val Bin F1: 0.8237


  0%|          | 0/352 [00:00<?, ?it/s]

  Epoch 10 | 188.8s | Train Loss: 0.2661 | Val Subtype F1: 0.1469 | Val Bin F1: 0.8625


  0%|          | 0/352 [00:00<?, ?it/s]

# Section 12: Cross-Fold Results Aggregation

In [ ]:
# ============================================================
# Section 12: Cross-Fold Results Aggregation
# ============================================================

def aggregate_results(results):
    metric_keys = [
        'binary_accuracy', 'binary_macro_f1', 'binary_balanced_accuracy', 'binary_mcc',
        'subtype_accuracy', 'subtype_macro_f1', 'subtype_weighted_f1', 'subtype_balanced_accuracy', 'subtype_mcc'
    ]
    summary = {}
    for key in metric_keys:
        values = [r['test_metrics'][key] for r in results]
        values = np.array(values, dtype=float)
        summary[key] = {
            'mean': float(values.mean()),
            'std': float(values.std()),
            'min': float(values.min()),
            'max': float(values.max()),
        }
    return summary

aggregated_summary = aggregate_results(all_fold_results)

print("=" * 75)
print("  OMNet-V3 Cross-Fold Aggregated Performance (Mean +/- Std across 5 Folds)")
print("=" * 75)
for k, stats in aggregated_summary.items():
    print(f"  {k:<30s}: {stats['mean']:.4f} +/- {stats['std']:.4f} (range: [{stats['min']:.4f}, {stats['max']:.4f}])")
print("=" * 75)

with open(OUTPUT_DIR / 'aggregated_results.json', 'w') as f:
    json.dump(aggregated_summary, f, indent=2)
print(f"[SAVE] Exported aggregated_results.json to {OUTPUT_DIR}")

# Section 13: Confusion Matrices (Binary & 8-Subtype)

In [ ]:
# ============================================================
# Section 13: Confusion Matrices (Binary & 8-Subtype)
# ============================================================

def plot_confusion_matrix(cm, labels, title, fname):
    fig, ax = plt.subplots(figsize=(7, 6))
    cm_arr = np.array(cm)
    cm_norm = cm_arr.astype(float) / np.maximum(cm_arr.sum(axis=1, keepdims=True), 1)
    labels_annot = np.array([f'{c}\n({p:.1%})' for c, p in zip(cm_arr.flatten(), cm_norm.flatten())]).reshape(cm_arr.shape)
    sns.heatmap(cm_arr, annot=labels_annot, fmt='', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    plt.tight_layout()
    plt.savefig(fname, dpi=150)
    plt.show()

# Aggregate confusion matrices across folds
cm_binary_total = np.zeros((2, 2), dtype=int)
cm_subtype_total = np.zeros((8, 8), dtype=int)

for res in all_fold_results:
    cm_binary_total += np.array(res['test_metrics']['binary_confusion_matrix'])
    cm_subtype_total += np.array(res['test_metrics']['subtype_confusion_matrix'])

plot_confusion_matrix(cm_binary_total, CONFIG['binary_classes'], 'Aggregated Binary Confusion Matrix (All Folds)', str(OUTPUT_DIR / 'confusion_matrix_binary.png'))
plot_confusion_matrix(cm_subtype_total, CONFIG['subtype_order'], 'Aggregated 8-Subtype Confusion Matrix (All Folds)', str(OUTPUT_DIR / 'confusion_matrix_8class.png'))

# Section 14: ROC & Precision-Recall Curves

In [ ]:
# ============================================================
# Section 14: ROC & Precision-Recall Curves
# ============================================================

all_bin_probs, all_bin_trues = [], []
all_sub_probs, all_sub_trues = [], []

for res in all_fold_results:
    all_bin_probs.extend([p[1] for p in res['test_metrics']['binary_probs']])
    all_bin_trues.extend(res['test_metrics']['binary_true'])
    all_sub_probs.extend(res['test_metrics']['subtype_probs'])
    all_sub_trues.extend(res['test_metrics']['subtype_true'])

all_bin_probs = np.array(all_bin_probs)
all_bin_trues = np.array(all_bin_trues)
all_sub_probs = np.array(all_sub_probs)
all_sub_trues = np.array(all_sub_trues)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Binary ROC
fpr, tpr, _ = roc_curve(all_bin_trues, all_bin_probs)
bin_auc = roc_auc_score(all_bin_trues, all_bin_probs)
axes[0].plot(fpr, tpr, label=f'Binary (AUC={bin_auc:.4f})', color='#e74c3c', linewidth=2.5)
axes[0].plot([0, 1], [0, 1], linestyle='--', color='gray')
axes[0].set_title('Binary ROC Curve (Aggregated Folds)', fontweight='bold')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(loc='lower right')

# Subtype OvR ROC Curves
for i, subtype in enumerate(CONFIG['subtype_order']):
    sub_true_binary = (all_sub_trues == i).astype(int)
    if sub_true_binary.sum() > 0:
        sub_fpr, sub_tpr, _ = roc_curve(sub_true_binary, all_sub_probs[:, i])
        sub_auc = roc_auc_score(sub_true_binary, all_sub_probs[:, i])
        axes[1].plot(sub_fpr, sub_tpr, label=f'{subtype} (AUC={sub_auc:.3f})')

axes[1].plot([0, 1], [0, 1], linestyle='--', color='gray')
axes[1].set_title('Subtype OvR ROC Curves', fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
save_figure(fig, 'roc_curves.png')
plt.show()

# Section 15: Fusion Gate Analysis across Magnifications

In [ ]:
# ============================================================
# Section 15: Fusion Gate Analysis across Magnifications
# ============================================================

best_fold_model = OMNetV3().to(DEVICE)
best_fold_ckpt = torch.load(OUTPUT_DIR / 'fold_0' / 'best_model.pth', map_location=DEVICE, weights_only=False)
best_fold_model.load_state_dict(best_fold_ckpt)
best_fold_model.eval()

_, _, test_loader_f0 = build_dataloaders(
    metadata.iloc[split_indices[0][0]],
    metadata.iloc[split_indices[0][0][:10]],
    metadata.iloc[split_indices[0][1]]
)

alpha_by_mag = {m: [] for m in CONFIG['magnification_levels']}
with torch.no_grad():
    for batch in test_loader_f0:
        images = batch['image'].to(DEVICE)
        mag_idx = batch['magnification_index'].to(DEVICE)
        outputs = best_fold_model(images, mag_idx)
        alphas = outputs['alpha'].detach().cpu().flatten().numpy()
        for i, m_idx in enumerate(batch['magnification_index'].numpy()):
            mag_val = CONFIG['magnification_levels'][m_idx]
            alpha_by_mag[mag_val].append(alphas[i])

print("=" * 60)
print("  Fusion Gate (Alpha) Distribution by Magnification")
print("=" * 60)
for mag, alphas in alpha_by_mag.items():
    if alphas:
        print(f"  {mag:3d}X: mean alpha = {np.mean(alphas):.4f} +/- {np.std(alphas):.4f} (CNN vs ViT balance)")
print("=" * 60)

fig, ax = plt.subplots(figsize=(8, 5))
data_for_plot = [[mag, a] for mag, alphas in alpha_by_mag.items() for a in alphas]
df_gate = pd.DataFrame(data_for_plot, columns=['Magnification', 'Alpha'])
sns.boxplot(data=df_gate, x='Magnification', y='Alpha', ax=ax, palette='Blues')
ax.set_title('Adaptive Fusion Gate Alpha across Magnification Levels', fontweight='bold')
ax.set_ylabel('Alpha (Weight on CNN branch)')
plt.tight_layout()
save_figure(fig, 'fusion_gate_analysis.png')
plt.show()

# Section 16: t-SNE Embedding Visualization & Grad-CAM

In [ ]:
# ============================================================
# Section 16: t-SNE Embedding Visualization & Grad-CAM
# ============================================================

all_features, all_sub_lbls = [], []
with torch.no_grad():
    for batch in test_loader_f0:
        images = batch['image'].to(DEVICE)
        mag_idx = batch['magnification_index'].to(DEVICE)
        outputs = best_fold_model(images, mag_idx)
        all_features.append(outputs['f_out'].cpu().numpy())
        all_sub_lbls.extend(batch['subtype_label'].numpy())

all_features = np.concatenate(all_features, axis=0)
all_sub_lbls = np.array(all_sub_lbls)

tsne = TSNE(n_components=2, random_state=CONFIG['seed'], perplexity=min(30, len(all_features)-1))
emb_2d = tsne.fit_transform(all_features)

fig, ax = plt.subplots(figsize=(9, 8))
for i, subtype in enumerate(CONFIG['subtype_order']):
    mask = (all_sub_lbls == i)
    if mask.sum() > 0:
        ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1], label=subtype, alpha=0.7, s=40)
ax.set_title('t-SNE of OMNet-V3 Fused Embeddings by Subtype', fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
save_figure(fig, 'tsne_embeddings.png')
plt.show()

# Section 17: Experiment Summary & Export

In [ ]:
# ============================================================
# Section 17: Experiment Summary & Export
# ============================================================

experiment_record = {
    'timestamp': datetime.now().isoformat(),
    'config': CONFIG,
    'aggregated_results': aggregated_summary,
    'folds_completed': len(all_fold_results),
}

with open(OUTPUT_DIR / 'experiment_config.json', 'w') as f:
    json.dump(experiment_record, f, indent=2, default=str)

df_summary = pd.DataFrame(aggregated_summary).T
df_summary.to_csv(OUTPUT_DIR / 'metrics_summary.csv')

print("=" * 75)
print("  OMNet-V3 Final Execution Summary")
print("=" * 75)
print(f"  Dataset             : BreakHis ({CONFIG['dataset_id']})")
print(f"  Magnifications      : {CONFIG['magnification_levels']}")
print(f"  Folds Evaluated     : {CONFIG['n_splits']}")
print(f"  Subtype Macro-F1    : {aggregated_summary['subtype_macro_f1']['mean']:.4f} +/- {aggregated_summary['subtype_macro_f1']['std']:.4f}")
print(f"  Binary Accuracy     : {aggregated_summary['binary_accuracy']['mean']:.4f} +/- {aggregated_summary['binary_accuracy']['std']:.4f}")
print(f"  Output Directory    : {OUTPUT_DIR}")
print("=" * 75)
print("[OK] All fold checkpoints, metrics, and figures saved successfully.")